# datalake NYC Taxi × Météo

## Contexte

La NYC Taxi & Limousine Commission (TLC) publie chaque mois, depuis 2009, l'intégralité des courses de taxis et de véhicules de transport avec chauffeur effectuées à New York. Ce jeu de données est massif et a etait modifier plusieur fois a travers les années 


---
## Les quatre types de véhicules

| Type | Début de publication | Remarque |
|---|---|---|
| **Yellow taxi** | Janvier 2009 | Le plus ancien, le plus documenté |
| **Green taxi** | Août 2013 | Créé pour desservir les zones hors Manhattan central |
| **FHV** (For-Hire Vehicle) | 2015 | Véhicules privé, ensuite uniquement Véhicules de tourisme/limousines |
| **FHVHV** (High Volume For-Hire Vehicle) | Février 2019 | Créé par la loi locale 149 (2018) pour les plateformes dispatchant plus de 10 000 courses/jour (Uber et Lyft) |

**Point d'attention :** depuis 2019 FHV capte seulement les courses des plateformes qui dépassent le seuil des 10 000 courses/jour, tandis que FHV continue de couvrir le reste des bases de VTC/limousines. FHV est structurellement beaucoup plus pauvre : il ne contient ni tarif, ni distance (sub based service)


## Localisation des prises en charge / dépose

Jusqu'en juin 2016, les fichiers contiennent des **coordonnées GPS brutes** (latitude/longitude) pour la prise en charge et la dépose. À partir de juillet 2016, la TLC ne publie plus de coordonnées GPS : chaque course référence à la place un **identifiant de zone** (`LocationID`), qui renvoie vers une table de référence publiée séparément par la TLC (zones, arrondissements, géométries).

## Colonnes apparues au fil du temps

Plusieurs colonnes tarifaires sont apparues progressivement dans le schéma (elles n'existaient tout simplement pas dans les fichiers plus anciens) :

- des frais aéroport
- un supplément d'amélioration du service, introduit en 2015 ;
- un supplément de congestion (congestion pricing), introduit en 2019 ;
- un nouveau supplément lié à la zone de tarification de congestion de Manhattan (« CBD »), en vigueur depuis le 5 janvier 2025.

## Fréquence de publication

Les données sont publiées mensuellement, avec un délai d'environ deux mois (les données de janvier sortent typiquement fin février/début mars).

---
# Bronze : ingestion et organisation dans HDFS

1. **Concevez une convention de nommage des chemins** dans HDFS pour organiser les données brutes. On ne vous dit pas quelle convention utiliser — mais votre convention doit permettre de répondre facilement, sans avoir à lire le contenu des fichiers, aux questions suivantes :
   - Quels types de véhicules ai-je déjà ingérés ?
   - Pour quelle période (année/mois) ai-je des données pour un type donné ?
   - Est-ce qu'un mois précis d'un type précis a déjà été ingéré (pour ne pas le re-télécharger) ?

2. **Écrivez le script d'ingestion** qui télécharge les fichiers et les dépose dans HDFS. Points critique :
   - il doit pouvoir être interrompu et relancé sans tout retélécharger ;
   - il doit gérer le fait que chaque type de véhicule n'existe pas depuis la même date;

3. **Ingérez aussi la météo.** source de données météo historiques horaires pour New York (Open-Meteo ou de la NOAA) le type de persitence pour les données meteroloique devrait etre logique par rapport a la source utilisé (API / parquet).


In [4]:
# Ingestion Bronze — les 4 types de véhicules TLC (yellow/green/fhv/fhvhv).
# Le script utilise EN PRIORITE le dossier nyc_taxi_data déjà fourni (monté
# en lecture seule sur /local_source, voir .env : NYC_TAXI_DATA_DIR) —
# aucun accès réseau nécessaire pour ce périmètre. Le réseau (TLC
# CloudFront) ne sert que si un fichier demandé n'est pas trouvé en local.
# Idempotent (marker _SUCCESS par partition HDFS) : interruption/relance
# sans tout refaire. Périmètre par défaut = exactement les 4 années x 6
# mois fournis localement (2009/2016/2019/2025, 01 a 06).
!python /home/jovyan/work/pipeline/ingestion/taxi_ingest.py --vehicle-type all

2026-08-25 17:00:13,677 INFO Instantiated <InsecureClient(url='http://namenode:9870')>.
2026-08-25 17:00:13,678 INFO Source locale : /local_source (montée)
2026-08-25 17:00:13,679 INFO Fetching status for '/bronze/taxi/vehicle_type=yellow/year=2009/month=01/_SUCCESS'.
2026-08-25 17:00:17,924 INFO Ingestion yellow 2009-01 depuis local:/local_source/2009/01/yellow_tripdata_2009-01.parquet...
2026-08-25 17:00:18,162 INFO Creating directories to '/bronze/taxi/vehicle_type=yellow/year=2009/month=01'.
2026-08-25 17:00:18,186 INFO Uploading '/tmp/nyc_taxi_cache/yellow_tripdata_2009-01.parquet' to '/bronze/taxi/vehicle_type=yellow/year=2009/month=01/yellow_tripdata_2009-01.parquet'.
2026-08-25 17:00:18,186 INFO Listing '/bronze/taxi/vehicle_type=yellow/year=2009/month=01/yellow_tripdata_2009-01.parquet'.
2026-08-25 17:00:18,191 INFO Writing to '/bronze/taxi/vehicle_type=yellow/year=2009/month=01/yellow_tripdata_2009-01.parquet'.
2026-08-25 17:00:21,452 INFO Writing to '/bronze/taxi/vehicle_typ

In [5]:
# Référentiel des zones TLC (nécessaire pour réconcilier GPS <-> LocationID
# en Silver) : téléchargement des géométries + construction de la grille
# de correspondance grid -> LocationID (une seule fois, idempotent).
!python /home/jovyan/work/pipeline/ingestion/zones_ingest.py
!python /home/jovyan/work/pipeline/ingestion/build_zone_grid.py

# Ingestion Bronze de la météo horaire NYC (Open-Meteo Historical API),
# alignée par défaut sur le même périmètre que le taxi (mêmes 4 années x
# 6 mois) — --full pour ingérer tout l'historique 2009-aujourd'hui.
!python /home/jovyan/work/pipeline/ingestion/weather_ingest.py

2026-08-25 17:08:24,341 INFO Instantiated <InsecureClient(url='http://namenode:9870')>.
2026-08-25 17:08:24,341 INFO Fetching status for '/bronze/reference/taxi_zones/_SUCCESS'.
2026-08-25 17:08:24,352 INFO Creating directories to '/bronze/reference/taxi_zones'.
2026-08-25 17:08:24,360 INFO Téléchargement https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv...
2026-08-25 17:08:24,543 INFO Uploading '/tmp/nyc_taxi_cache/taxi_zone_lookup.csv' to '/bronze/reference/taxi_zones/taxi_zone_lookup.csv'.
2026-08-25 17:08:24,543 INFO Listing '/bronze/reference/taxi_zones/taxi_zone_lookup.csv'.
2026-08-25 17:08:24,547 INFO Writing to '/bronze/reference/taxi_zones/taxi_zone_lookup.csv'.
2026-08-25 17:08:24,576 INFO Téléchargement https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip...
2026-08-25 17:08:24,682 INFO Uploading '/tmp/nyc_taxi_cache/taxi_zones.zip' to '/bronze/reference/taxi_zones/taxi_zones.zip'.
2026-08-25 17:08:24,682 INFO Listing '/bronze/reference/taxi_zones/taxi_zo

---
# Silver : nettoyage, réconciliation et modélisation


## Choisissez votre système de stockage



### Réconcilier les deux systèmes de localisation

Rappel : coordonnées GPS brutes avant juillet 2016, identifiants de zone après. Vous devez faire en sorte que toutes vos courses, quelle que soit leur date, puissent être rattachées à une même notion de zone/arrondissement.


### Gérer les colonnes apparues progressivement

Rappel : plusieurs suppléments tarifaires n'existaient pas dans les fichiers les plus anciens. Une valeur manquante *avant* l'introduction d'un supplément et une valeur de 0 *après* son introduction ne signifient pas la même chose. Assurez-vous que votre pipeline ne confond pas les deux pour bien les prendre en consideration plus tard au niveau des insights



votre modèle silver doit permettre de répondre, sans retourner aux fichiers bronze, à des questions comme : 

*combien de courses par type de véhicule et par mois*, 
*quelle est la zone de prise en charge la plus fréquente*, 
*quelle est la météo au moment d'une course donnée*. 

### modélisation silver

Une table `/silver/taxi` (Parquet, partitionné par `vehicle_type/year/month`)
avec un schéma canonique commun aux 4 types de véhicules : datetimes de
pickup/dropoff, `pickup_zone_id`/`dropoff_zone_id` (unifiés GPS/LocationID),
distance, tarifs de base, et un couple `(valeur, disponible)` par supplément
progressif — `disponible=False` signifiant explicitement "n'existait pas
encore dans le schéma source ce mois-là", à ne jamais confondre avec une
vraie valeur 0. Une table `/silver/weather` (partitionnée `year/month`) à
côté, jointe à la météo horaire au moment du pickup.

Cela répond directement aux 3 questions du sujet sans retourner à Bronze :
courses par type/mois (`GROUP BY vehicle_type, year, month`), zone de
pickup la plus fréquente (`GROUP BY pickup_zone_id`), météo au moment
d'une course (déjà jointe dans la table).

In [ ]:
# Job Silver : cluster réel (spark://spark-master:7077), jamais local[*],
# conformément à l'exigence du cours. La logique de réconciliation
# (GPS <-> LocationID via la grille pré-calculée, distinction NULL vs 0
# pour les suppléments apparus progressivement) est documentée en tête de
# transform_silver.py — le point clé à retenir : on lit CHAQUE partition
# Bronze individuellement (jamais un mergeSchema global sur tout
# l'historique) pour pouvoir constater, mois par mois, si une colonne de
# supplément existe réellement avant de décider NULL ou 0.
!spark-submit --master spark://spark-master:7077 --conf spark.executor.memory=3g \
    /home/jovyan/work/pipeline/silver/transform_silver.py

---
# Gold : des tables construites pour des questions précises

partez de la question, et construisez la table (ou la requête) qui y répond directement.

Voici les quatre analyses que votre datalake doit permettre.

## 1. Évolution des nouveaux suppléments tarifaires

**Objectif :** pour chaque supplément apparu progressivement dans le schéma, montrer son évolution dans le temps (montant total ou moyen collecté), par type de véhicule.

## 2. Flux de prises en charge / déposes (diagramme en corde)

**Objectif :** visualiser, sous forme de chord diagram, les flux de courses entre zones de départ et d'arrivée.

## 3. Évolution du prix des courses par zone de départ (ridgeline plot)

**Objectif :** un ridgeline plot montrant l'evolution des prix par km au fil des années, pour une zone de départ donnée.

## 4. Fréquence des courses par heure et jour de la semaine (heatmap)

**Objectif :** une fonction qui, pour une année donnée en paramètre, produit une heatmap (jour de la semaine × heure de la journée) de la fréquence des courses, pour chaque type de véhicule.

## 5. Écart de prix selon la météo

**Objectif :** comparer le prix des courses (par km) selon les conditions météo au moment de la course, par rapport au prix moyen général 

## À vous de jouer

Pour chacune des quatre analyses : concevez la ou les tables gold nécessaires (ou la requête directe sur silver, si vous jugez qu'une table gold dédiée n'est pas nécessaire !!!!!!) 

In [ ]:
# Exercice : Évolution des nouveaux suppléments tarifaires
!spark-submit --master spark://spark-master:7077 /home/jovyan/work/pipeline/gold/gold_surcharges.py

import sys
sys.path.append("/home/jovyan/work/pipeline")
import pandas as pd
import plotly.express as px
from notebook_utils import read_gold_table

surcharges = read_gold_table("/gold/surcharges_evolution", refresh=True)
surcharges["date"] = pd.to_datetime(dict(year=surcharges.year, month=surcharges.month, day=1))

fig = px.line(
    surcharges.sort_values("date"),
    x="date", y="montant_moyen", color="vehicle_type",
    facet_row="supplement",
    title="Évolution des suppléments tarifaires par type de véhicule",
    labels={"montant_moyen": "Montant moyen ($)", "date": "Mois"},
)
fig.update_yaxes(matches=None)
fig.show()

In [ ]:
# Exercice : Diagramme en corde des flux pickup/dropoff
!spark-submit --master spark://spark-master:7077 /home/jovyan/work/pipeline/gold/gold_flows.py

import holoviews as hv
hv.extension("bokeh")

flows_borough = read_gold_table("/gold/flows_by_borough", refresh=True)
flows_borough = flows_borough.dropna(subset=["pickup_borough", "dropoff_borough"])
agg = flows_borough.groupby(["pickup_borough", "dropoff_borough"])["nb_courses"].sum().reset_index()
# on garde les flux INTER-arrondissement (intra-arrondissement écraserait
# visuellement le reste, Manhattan->Manhattan étant très majoritaire)
agg = agg[agg["pickup_borough"] != agg["dropoff_borough"]]

boroughs = sorted(set(agg["pickup_borough"]) | set(agg["dropoff_borough"]))
index_of = {b: i for i, b in enumerate(boroughs)}
nodes = hv.Dataset(pd.DataFrame({"index": range(len(boroughs)), "name": boroughs}), "index")
links = pd.DataFrame({
    "source": agg["pickup_borough"].map(index_of),
    "target": agg["dropoff_borough"].map(index_of),
    "value": agg["nb_courses"],
})

chord = hv.Chord((links, nodes)).opts(
    node_color="name", cmap="Category10", edge_color="source", edge_cmap="Category10",
    labels="name", width=600, height=600,
    title="Flux de courses entre arrondissements (pickup -> dropoff)",
)
chord

In [ ]:
# Exercice : Ridgeline plot de l'évolution du prix par zone de départ
!spark-submit --master spark://spark-master:7077 /home/jovyan/work/pipeline/gold/gold_price_ridgeline.py

import joypy
import matplotlib.pyplot as plt

price_data = read_gold_table("/gold/price_per_km_by_zone_year", refresh=True)

# Zone de départ à visualiser : paramétrable, ici on prend celle qui a le
# plus de lignes échantillonnées pour garantir une distribution lisible
# sur toutes les années.
zone_id = price_data["pickup_zone_id"].value_counts().idxmax()
subset = price_data[price_data["pickup_zone_id"] == zone_id]

fig, axes = joypy.joyplot(
    subset, by="year", column="price_per_km",
    figsize=(8, 6), overlap=2, colormap=plt.cm.viridis,
    title=f"Évolution du prix par km — zone de départ {zone_id}",
)
plt.xlabel("Prix par km ($)")
plt.show()

In [ ]:
# Exercice : Fonction heatmap(annee) : fréquence des courses par jour/heure et par type de véhicule
!spark-submit --master spark://spark-master:7077 /home/jovyan/work/pipeline/gold/gold_heatmap.py

import seaborn as sns
import matplotlib.pyplot as plt

_heatmap_data = read_gold_table("/gold/rides_by_dow_hour", refresh=True)
_DOW_LABELS = {1: "Dim", 2: "Lun", 3: "Mar", 4: "Mer", 5: "Jeu", 6: "Ven", 7: "Sam"}

def heatmap(annee, vehicle_type=None):
    """Heatmap jour de la semaine x heure de la journée de la fréquence des
    courses, pour l'année donnée. Si vehicle_type est précisé, filtre sur
    ce seul type ; sinon agrège les 4 types ensemble."""
    data = _heatmap_data[_heatmap_data["year"] == annee]
    if vehicle_type:
        data = data[data["vehicle_type"] == vehicle_type]
    pivot = (
        data.groupby(["day_of_week", "hour_of_day"])["nb_courses"].sum()
        .unstack(fill_value=0)
        .reindex(index=range(1, 8), columns=range(24), fill_value=0)
    )
    pivot.index = [_DOW_LABELS[i] for i in pivot.index]

    plt.figure(figsize=(12, 5))
    title = f"Fréquence des courses par jour/heure — {annee}"
    if vehicle_type:
        title += f" ({vehicle_type})"
    sns.heatmap(pivot, cmap="YlOrRd", cbar_kws={"label": "Nombre de courses"})
    plt.title(title)
    plt.xlabel("Heure de la journée")
    plt.ylabel("Jour de la semaine")
    plt.show()
    return pivot

# Exemple d'appel :
heatmap(2019, vehicle_type="yellow")

### 5. Écart de prix selon la météo

*(cellule ajoutée : le sujet liste 5 analyses Gold dans son énoncé mais ce
notebook ne fournissait que 4 cellules "À COMPLÉTER" — celle-ci comble
l'écart pour couvrir intégralement les 5 analyses demandées.)*

In [ ]:
# Exercice : Écart de prix selon la météo
!spark-submit --master spark://spark-master:7077 /home/jovyan/work/pipeline/gold/gold_weather_price.py

weather_price = read_gold_table("/gold/price_by_weather", refresh=True)

fig = px.bar(
    weather_price.sort_values("ecart_pct"),
    x="weather_bucket", y="ecart_pct", color="vehicle_type", barmode="group",
    title="Écart de prix par km selon la météo, vs. moyenne générale",
    labels={"ecart_pct": "Écart au prix moyen général (%)", "weather_bucket": "Condition météo"},
)
fig.show()
weather_price